In [1]:
from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder, find_peaks
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Paths
cutout_dir = Path("../data/raw/v854_cen_dasch/cutouts")
output_dir = Path("../outputs/tables")
os.makedirs(output_dir, exist_ok=True)

# Detection parameters
FWHM = 3.0
THRESHOLD_SIGMA = 5.0
MIN_SOURCES_DAO = 5   # fall back to find_peaks if DAOStarFinder finds fewer than this
BOX_SIZE = 11         # for find_peaks

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts to process")

Found 4540 cutouts to process


In [2]:
def detect_sources(fits_path):
    data = fits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))

    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    data_sub = data - median

    try:
        daofind = DAOStarFinder(
            fwhm=FWHM,
            threshold=THRESHOLD_SIGMA * std,
            sharpness_range=(0.2, 2.0)
        )
        dao_sources = daofind(data_sub)
        n_dao = 0 if dao_sources is None else len(dao_sources)
    except Exception:
        dao_sources = None
        n_dao = 0

    if n_dao >= MIN_SOURCES_DAO:
        sources = dao_sources
        algorithm = "DAOStarFinder"
        x_col, y_col = "x_centroid", "y_centroid"
    else:
        sources = find_peaks(data_sub, threshold=THRESHOLD_SIGMA * std, box_size=BOX_SIZE)
        algorithm = "find_peaks"
        x_col, y_col = "x_peak", "y_peak"

        if sources is not None and len(sources) > 0:
            from photutils.morphology import data_properties
            sharpness_vals, elongation_vals = [], []
            for row in sources:
                x, y = int(row["x_peak"]), int(row["y_peak"])
                cutout = data_sub[max(0,y-7):y+7, max(0,x-7):x+7]
                if cutout.size > 0:
                    try:
                        props = data_properties(cutout)
                        sharpness_vals.append(float(props.max_value / props.segment_fluxes[0]) if props.segment_fluxes[0] > 0 else np.nan)
                        elongation_vals.append(float(props.elongation))
                    except:
                        sharpness_vals.append(np.nan)
                        elongation_vals.append(np.nan)
                else:
                    sharpness_vals.append(np.nan)
                    elongation_vals.append(np.nan)
            sources["sharpness"] = sharpness_vals
            sources["elongation"] = elongation_vals

    return sources, algorithm, data, data_sub, std, x_col, y_col

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

import pickle
from pathlib import Path

from photutils.aperture import CircularAperture, aperture_photometry

from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, match_coordinates_sky
import astropy.units as u

from astroquery.gaia import Gaia
from astroquery.simbad import Simbad

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import ndimage
from skimage.transform import hough_line, hough_line_peaks
import warnings

APERTURE_RADIUS = 5.0

# Target
V854_CEN = SkyCoord(
    ra=218.70584352985 * u.deg,
    dec=-39.55533369119 * u.deg
)

# Cache

BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")

gaia_dir = BASE_DIR / "data" / "gaia"
gaia_dir.mkdir(parents=True, exist_ok=True)

GAIA_CACHE_FILE = gaia_dir / "gaia_cache.pkl"
NAME_CACHE_FILE = gaia_dir / "gaia_name_cache.pkl"

if GAIA_CACHE_FILE.exists():
    with open(GAIA_CACHE_FILE, "rb") as f:
        gaia_cache = pickle.load(f)
else:
    gaia_cache = {}

if NAME_CACHE_FILE.exists():
    with open(NAME_CACHE_FILE, "rb") as f:
        gaia_name_cache = pickle.load(f)
else:
    gaia_name_cache = {}

# Display mode
display_mode = widgets.ToggleButtons(
    options=["GAIA ID", "Object Name"],
    value="GAIA ID",
    description="Labels:"
)

# Error toggle — single show/hide
error_toggle = widgets.ToggleButtons(
    options=["Show Errors", "Hide Errors"],
    value="Hide Errors",
    description="Errors:"
)

# Metadata
def get_plate_date(fits_path):
    try:
        header = astrofits.getheader(fits_path)
        for key in ["DATE-OBS", "DATE", "DATEOBS", "MJD-OBS"]:
            if key in header:
                return str(header[key])
        return "Unknown"
    except:
        return "Unknown"

# Error detection
def detect_plate_errors(data):

    errors = {
        "scratches":  [],
        "trailing":   [],
        "saturation": [],
        "dust":       [],
        "edge":       False,
    }

    h, w = data.shape

    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    if hi == lo:
        return errors
    norm = np.clip((data - lo) / (hi - lo), 0, 1)

    # Scratches
    try:
        scale   = 4
        small   = ndimage.zoom(norm, 1 / scale, order=1)
        sobel_h = ndimage.sobel(small, axis=0)
        sobel_v = ndimage.sobel(small, axis=1)
        edges   = np.hypot(sobel_h, sobel_v)
        edges   = (edges > np.percentile(edges, 97)).astype(np.uint8)

        tested_angles = np.linspace(-np.pi / 2, np.pi / 2, 180, endpoint=False)
        hspace, angles, dists = hough_line(edges, theta=tested_angles)

        peaks = hough_line_peaks(
            hspace, angles, dists,
            num_peaks=8,
            min_distance=20,
            threshold=0.35 * hspace.max()
        )

        sh, sw = small.shape
        for _, angle, dist in zip(*peaks):
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            if abs(sin_a) > 1e-6:
                x0_s, x1_s = 0, sw - 1
                y0_s = (dist - x0_s * cos_a) / sin_a
                y1_s = (dist - x1_s * cos_a) / sin_a
            else:
                y0_s, y1_s = 0, sh - 1
                x0_s = x1_s = dist / cos_a if abs(cos_a) > 1e-6 else 0

            y0_s = float(np.clip(y0_s, 0, sh - 1))
            y1_s = float(np.clip(y1_s, 0, sh - 1))
            x0_s = float(np.clip(x0_s, 0, sw - 1))
            x1_s = float(np.clip(x1_s, 0, sw - 1))

            errors["scratches"].append({
                "x0": x0_s * scale, "y0": y0_s * scale,
                "x1": x1_s * scale, "y1": y1_s * scale,
            })
    except Exception as e:
        print(f"[SCRATCH DETECT] {e}")

    # Trailing stars — only flag sources with extreme elongation (>5) and minimum area
    try:
        bright_mask = (norm > np.percentile(norm, 95)).astype(np.uint8)
        labeled, n_obj = ndimage.label(bright_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 40 or area > 0.005 * h * w:
                continue

            coords = np.argwhere(region)
            if len(coords) < 10:
                continue

            cov  = np.cov(coords[:, 1], coords[:, 0])
            eigs = np.linalg.eigvalsh(cov)
            eigs = np.sort(np.abs(eigs))
            if eigs[0] < 1e-6:
                continue

            elongation = np.sqrt(eigs[1] / eigs[0])
            if elongation > 5.0:
                cy_r, cx_r = coords.mean(axis=0)
                angle = 0.5 * np.degrees(np.arctan2(
                    2 * cov[0, 1], cov[0, 0] - cov[1, 1]
                ))
                major = 2 * np.sqrt(eigs[1])
                minor = 2 * np.sqrt(eigs[0])
                errors["trailing"].append({
                    "x": float(cx_r), "y": float(cy_r),
                    "w": float(major), "h": float(minor),
                    "angle": float(angle)
                })
    except Exception as e:
        print(f"[TRAIL DETECT] {e}")

    # Saturation / blooming — must be large and at the absolute ceiling
    try:
        sat_thresh = np.percentile(norm, 99.9)
        sat_mask   = (norm >= sat_thresh).astype(np.uint8)
        sat_mask   = ndimage.binary_dilation(sat_mask, iterations=2).astype(np.uint8)
        labeled, n_obj = ndimage.label(sat_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 200:
                continue

            coords  = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:, 0].max() - coords[:, 0].min()) / 2
            rx = (coords[:, 1].max() - coords[:, 1].min()) / 2
            r  = np.hypot(rx, ry)

            errors["saturation"].append({
                "x": float(cx_r), "y": float(cy_r), "r": float(r)
            })
    except Exception as e:
        print(f"[SAT DETECT] {e}")

    # Dust spots
    try:
        dust_thresh = np.percentile(norm, 2)
        dark_mask   = (norm <= dust_thresh).astype(np.uint8)
        dark_mask   = ndimage.binary_erosion(dark_mask, iterations=2).astype(np.uint8)
        labeled, n_obj = ndimage.label(dark_mask)

        for obj_id in range(1, n_obj + 1):
            region = labeled == obj_id
            area   = region.sum()
            if area < 20 or area > 0.005 * h * w:
                continue

            coords  = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:, 0].max() - coords[:, 0].min()) / 2
            rx = (coords[:, 1].max() - coords[:, 1].min()) / 2

            if rx < 1e-6 or ry < 1e-6:
                continue
            aspect = max(rx, ry) / min(rx, ry)
            if aspect > 3:
                continue

            r = np.hypot(rx, ry)
            errors["dust"].append({
                "x": float(cx_r), "y": float(cy_r), "r": float(r)
            })
    except Exception as e:
        print(f"[DUST DETECT] {e}")

    # Edge artifacts
    try:
        border = max(20, int(min(h, w) * 0.05))
        center_med = np.nanmedian(norm[border:h-border, border:w-border])
        center_std = np.nanstd(norm[border:h-border, border:w-border])

        edge_strips = [
            norm[:border, :],
            norm[h-border:, :],
            norm[:, :border],
            norm[:, w-border:],
        ]

        for strip in edge_strips:
            strip_med = np.nanmedian(strip)
            if abs(strip_med - center_med) > 3 * center_std:
                errors["edge"] = True
                break
    except Exception as e:
        print(f"[EDGE DETECT] {e}")

    return errors


def annotate_errors(ax, errors):
    # Draw all error types at once, each with its own color and category label

    legend_handles = []

    # Scratches
    for s in errors["scratches"]:
        ax.plot(
            [s["x0"], s["x1"]], [s["y0"], s["y1"]],
            color="magenta", linewidth=1.2, alpha=0.8, linestyle="--"
        )
    if errors["scratches"]:
        legend_handles.append(
            Line2D([0], [0], color="magenta", linestyle="--", label=f'Scratch ×{len(errors["scratches"])}')
        )

    # Trailing stars
    for t in errors["trailing"]:
        ellipse = mpatches.Ellipse(
            (t["x"], t["y"]),
            width=t["w"], height=t["h"],
            angle=t["angle"],
            edgecolor="orange", facecolor="none",
            linewidth=1.5, alpha=0.85
        )
        ax.add_patch(ellipse)
        ax.text(
            t["x"] + t["w"] / 2 + 3, t["y"],
            "trail", color="orange", fontsize=5,
            va="center"
        )
    if errors["trailing"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="orange", facecolor="none",
                           label=f'Trailing ×{len(errors["trailing"])}')
        )

    # Saturation
    for s in errors["saturation"]:
        circ = plt.Circle(
            (s["x"], s["y"]), s["r"],
            edgecolor="red", facecolor="none",
            linewidth=1.5, alpha=0.8, linestyle="-."
        )
        ax.add_patch(circ)
        ax.text(
            s["x"], s["y"] - s["r"] - 4,
            "sat", color="red", fontsize=5,
            ha="center"
        )
    if errors["saturation"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="red", facecolor="none",
                           label=f'Saturation ×{len(errors["saturation"])}')
        )

    # Dust spots
    for d in errors["dust"]:
        circ = plt.Circle(
            (d["x"], d["y"]), d["r"],
            edgecolor="deepskyblue", facecolor="none",
            linewidth=1.2, alpha=0.8, linestyle=":"
        )
        ax.add_patch(circ)
        ax.text(
            d["x"], d["y"] - d["r"] - 4,
            "dust", color="deepskyblue", fontsize=5,
            ha="center"
        )
    if errors["dust"]:
        legend_handles.append(
            mpatches.Patch(edgecolor="deepskyblue", facecolor="none",
                           label=f'Dust ×{len(errors["dust"])}')
        )

    # Edge artifacts
    if errors["edge"]:
        ax_h, ax_w = ax.get_ylim(), ax.get_xlim()
        img_h = abs(ax_h[1] - ax_h[0])
        img_w = abs(ax_w[1] - ax_w[0])
        rect = mpatches.Rectangle(
            (min(ax_w), min(ax_h)), img_w, img_h,
            edgecolor="lime", facecolor="none",
            linewidth=2.5, alpha=0.7
        )
        ax.add_patch(rect)
        legend_handles.append(
            mpatches.Patch(edgecolor="lime", facecolor="none",
                           label="Edge artifact")
        )

    if legend_handles:
        ax.legend(
            handles=legend_handles,
            loc="upper left",
            fontsize=6,
            framealpha=0.6,
            facecolor="black",
            labelcolor="white",
            edgecolor="gray"
        )

# Retroactive label enrichment: re-resolve names for all already-scanned plates using whatever is now in gaia_name_cache (without re-querying Gaia or SIMBAD).
def enrich_previous_plates(plate_db, up_to_key):

    keys = list(plate_db.keys())
    stop = keys.index(up_to_key) if up_to_key in keys else len(keys)

    for f in keys[:stop]:

        r = plate_db[f].get("render")

        if r is None:
            continue

        old_resolved = r["resolved_names"]
        new_resolved = resolve_names(r["gaia_names"], ["Unknown"] * len(r["gaia_names"]))

        # Only update entries that were previously Unknown and are now known
        merged = []
        for old, new in zip(old_resolved, new_resolved):
            if old == "Unknown" and new != "Unknown":
                merged.append(new)
            else:
                merged.append(old)

        r["resolved_names"] = merged

# GAIA catalog
def get_plate_catalog_gaia(wcs, shape):

    cx, cy = shape[1] / 2, shape[0] / 2
    ra_c, dec_c = wcs.pixel_to_world_values(cx, cy)

    ra_c = float(np.array(ra_c))
    dec_c = float(np.array(dec_c))

    key = (round(ra_c, 4), round(dec_c, 4))

    if key in gaia_cache:
        return gaia_cache[key]

    radius = 0.25 * u.deg
    center = SkyCoord(ra_c * u.deg, dec_c * u.deg)

    query = f"""
    SELECT source_id, ra, dec, phot_g_mean_mag
    FROM gaiadr3.gaia_source
    WHERE CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra_c}, {dec_c}, {radius.to(u.deg).value})
    ) = 1
    """

    try:
        job = Gaia.launch_job_async(query)
        result = job.get_results()

        if result is None:
            result = []

        gaia_cache[key] = result

        with open(GAIA_CACHE_FILE, "wb") as f:
            pickle.dump(gaia_cache, f)
            
        print(f"[GAIA] loaded {len(result)} sources")

        return result

    except Exception as e:
        print("[GAIA ERROR]", e)
        gaia_cache[key] = []
        return []

# SIMBAD Name resolver
def get_simbad_names(ra, dec, max_sep_arcsec=5):

    names = ["Unknown"] * len(ra)

    try:
        simbad = Simbad()
        simbad.TIMEOUT = 60

        coords = SkyCoord(
            ra=np.asarray(ra) * u.deg,
            dec=np.asarray(dec) * u.deg
        )

        result = simbad.query_region(
            coords,
            radius=max_sep_arcsec * u.arcsec
        )

        if result is None:
            return names

        if len(result) == 0:
            return names

        sim_coords = SkyCoord(
            result["ra"],
            result["dec"],
            unit="deg"
        )

        if len(sim_coords) == 0:
            return names

        idx, sep, _ = match_coordinates_sky(
            coords,
            sim_coords
        )

        for i in range(len(coords)):
            if sep[i].arcsec < max_sep_arcsec:
                
                name = str(result["main_id"][idx[i]])
                # Hide Gaia identifiers in Object Name mode
                
                if (
                    name.startswith("Gaia DR3")
                    or name.startswith("Gaia DR2")
                    or name.startswith("Gaia DR1")
                    or name.startswith("GAIA")
                ):

                    names[i] = "Unknown"
                else:
                    names[i] = name

    except Exception as e:
        print("[SIMBAD]", e)

    return names

# GAIA Crossmatch labels
def match_detected_sources_gaia(ra, dec, catalog, max_sep_arcsec=5):

    if catalog is None or len(catalog) == 0:
        return ["Unknown"] * len(ra)

    cat_coords = SkyCoord(catalog["ra"], catalog["dec"], unit="deg")
    src_coords = SkyCoord(ra * u.deg, dec * u.deg)

    idx, sep, _ = match_coordinates_sky(src_coords, cat_coords)

    labels = []

    for i in range(len(src_coords)):
        if sep[i].arcsec <= max_sep_arcsec:
            labels.append(f"Gaia {catalog['source_id'][idx[i]]}")
        else:
            labels.append("Unknown")

    return labels

def update_name_cache(gaia_labels, simbad_names):

    changed = False

    for g, s in zip(gaia_labels, simbad_names):

        if s == "Unknown":
            continue

        if not g.startswith("Gaia "):
            continue

        try:
            source_id = int(g.replace("Gaia ", ""))

            gaia_name_cache[source_id] = s
            changed = True

        except:
            pass

    if changed:
        with open(NAME_CACHE_FILE, "wb") as f:
            pickle.dump(gaia_name_cache, f)


def resolve_names(gaia_labels, simbad_names):

    labels = []

    for g, s in zip(gaia_labels, simbad_names):

        if s != "Unknown":
            labels.append(s)
            continue

        if g.startswith("Gaia "):

            try:
                source_id = int(g.replace("Gaia ", ""))

                if source_id in gaia_name_cache:
                    labels.append(
                        gaia_name_cache[source_id]
                    )
                    continue

            except:
                pass

        labels.append("Unknown")

    return labels

# V854 CEN Match
def find_v854cen_match(ra, dec, max_sep_arcsec=10):

    src = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
    sep = src.separation(V854_CEN)

    idx = np.argmin(sep)
    best = sep[idx].arcsec

    return (best < max_sep_arcsec), idx, best

# Pre-compute sources, photometry, labels, and errors per plate
# After each plate, enrich all previous plates with any newly learned names
scan_progress = widgets.IntProgress(value=0, min=0, max=len(cutouts), description="Scanning:")
scan_status = widgets.Label(value="Scanning...")

display(widgets.VBox([scan_progress, scan_status]))

plate_db = {}

DEBUG_N = 30   # Change this for testing.

for i, f in enumerate(cutouts[:DEBUG_N]): # Change to enumerate(cutouts) to analyze all plates. Change to cutouts[:DEBUG_N] to iterate on DEBUG_N plates.

    scan_status.value = f.name

    try:
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(f)

        tag = "error"
        n = 0
        has_target = False
        cached_render = None

        if sources is not None and len(sources) > 0:

            n = len(sources)

            hdr = astrofits.getheader(f)
            wcs = WCS(hdr)

            ra, dec = wcs.pixel_to_world_values(
                sources[x_col],
                sources[y_col]
            )

            ra  = np.array(ra,  dtype=float)
            dec = np.array(dec, dtype=float)

            found, idx, sep = find_v854cen_match(ra, dec)
            has_target = found

            sharp = sources["sharpness"] if "sharpness" in sources.colnames else None
            elong = sources["elongation"] if "elongation" in sources.colnames else None

            bad = 0
            if sharp is not None:
                bad += np.sum(np.array(sharp) < 0.1)
            if elong is not None:
                bad += np.sum(np.array(elong) > 3)

            bad_frac = bad / n

            if bad_frac > 0.5:
                tag = "messy"
            elif n < 10:
                tag = "empty"
            elif n < 100:
                tag = "crowded"
            else:
                tag = "rich"

            # Photometry
            positions  = list(zip(sources[x_col], sources[y_col]))
            apertures  = CircularAperture(positions, r=APERTURE_RADIUS)
            phot_table = aperture_photometry(data_sub, apertures)
            flux       = np.array(phot_table["aperture_sum"])
            flux       = np.where(flux > 0, flux, np.nan)
            inst_mag   = -2.5 * np.log10(flux)
            # Add full-width sizes; use B magnitude to refine this. Sanity check in new cell to get a straight line

            # Catalog crossmatch
            gaia_catalog = get_plate_catalog_gaia(wcs, data.shape)
            gaia_names   = match_detected_sources_gaia(ra, dec, gaia_catalog)

            try:
                simbad_names = get_simbad_names(ra, dec)
            except Exception as e:
                print("[SIMBAD ERROR]", e)
                simbad_names = ["Unknown"] * len(ra)

            update_name_cache(gaia_names, simbad_names)
            resolved_names = resolve_names(gaia_names, simbad_names)

            # Error detection
            plate_errors = detect_plate_errors(data)

            cached_render = {
                "data":           data,
                "sources":        sources,
                "x_col":          x_col,
                "y_col":          y_col,
                "inst_mag":       inst_mag,
                "gaia_names":     gaia_names,
                "resolved_names": resolved_names,
                "found":          found,
                "target_idx":     idx,
                "errors":         plate_errors,
            }

    except:
        tag = "error"
        n = 0
        has_target = False
        cached_render = None

    plate_db[f] = {
        "tag":    tag,
        "n":      n,
        "date":   get_plate_date(f),
        "target": has_target,
        "render": cached_render,
    }

    # Retroactive enrichment (propagate any names learned from this plate back into earlier plates).
    enrich_previous_plates(plate_db, f)

    scan_progress.value = i + 1

scan_status.value = "Scan complete"

# UI
filter_dropdown = widgets.Dropdown(
    options=["all", "empty", "crowded", "rich", "messy", "error", "target"],
    value="all",
    description="Filter:"
)

dropdown = widgets.Dropdown(description="Plate:")

progress = widgets.IntProgress(value=0, min=0, max=100, description="Progress:")
status = widgets.Label(value="Ready")
output = widgets.Output()

# Rebuild dropdown
def rebuild_dropdown(*args):

    options = []

    for f, meta in plate_db.items():

        if filter_dropdown.value == "target" and not meta["target"]:
            continue

        if filter_dropdown.value not in ["all", "target"]:
            if meta["tag"] != filter_dropdown.value:
                continue

        label = f"{f.name} | {meta['tag']} ({meta['n']}) | V854:{meta['target']} | {meta['date']}"
        options.append((label, f))

    dropdown.options = options

filter_dropdown.observe(rebuild_dropdown, names="value")

# Viewer renders from cache only, no recomputation
def show_plate(change):

    fits_path = dropdown.value

    if fits_path is None:
        print("No plate selected.")
        return

    meta = plate_db.get(fits_path)

    if meta is None or meta["render"] is None:
        with output:
            clear_output(wait=True)
            print("No data available for this plate.")
        return

    r = meta["render"]

    progress.value = 50
    status.value = "Rendering..."

    with output:
        clear_output(wait=True)

        # Label switch
        if display_mode.value == "Object Name":
            labels = r["resolved_names"]
        else:
            labels = r["gaia_names"]

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(r["data"], origin="lower", cmap="gray")
        axes[0].set_title("Raw Plate")

        axes[1].imshow(r["data"], origin="lower", cmap="gray")
        axes[1].set_title("Source Detections")

        axes[1].scatter(
            r["sources"][r["x_col"]],
            r["sources"][r["y_col"]],
            s=50,
            facecolors="none",
            edgecolors="red"
        )

        if r["found"]:
            axes[1].scatter(
                r["sources"][r["x_col"]][r["target_idx"]],
                r["sources"][r["y_col"]][r["target_idx"]],
                s=120,
                edgecolors="cyan",
                facecolors="none"
            )

        for i, (x, y, mag) in enumerate(zip(
            r["sources"][r["x_col"]],
            r["sources"][r["y_col"]],
            r["inst_mag"]
        )):

            name = labels[i] if i < len(labels) else "Unknown"

            if np.isfinite(mag):
                axes[1].text(
                    x + 5,
                    y + 5,
                    f"{name}\n{mag:.2f}",
                    color="yellow",
                    fontsize=6,
                    bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
                )

        # Error overlay — all categories at once when shown
        if error_toggle.value == "Show Errors":
            annotate_errors(axes[1], r["errors"])

        plt.tight_layout()
        plt.show()

        progress.value = 100
        status.value = "Done"

# Initialization
rebuild_dropdown()

dropdown.observe(show_plate, names="value")
display_mode.observe(show_plate, names="value")
error_toggle.observe(show_plate, names="value")

display(widgets.VBox([
    display_mode,
    error_toggle,
    filter_dropdown,
    dropdown,
    progress,
    status,
    output
]))

if len(cutouts) > 0:
    dropdown.value = cutouts[0]